In [280]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

##Loading of The Dataset

In [281]:

# Load the actual Birmingham Parking Dataset
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/raw_car_dataset.csv')

## 1 Load & Inspect

In [282]:
print(df.head(10))

    Price  Odometer_km  Doors  Accidents Location  Year
0  $1,500     137830.0    4.0          1     City  1998
1  4171.0          NaN    4.0          0    Rural  2016
2  $5,331     107302.0    4.0          0   Suburb  2014
3  1500.0     141838.0    4.0          1   Suburb  1999
4  1500.0          NaN    3.0          0     City  2022
5  $1,500     211171.0    4.0          0       ??  2003
6  1500.0     222235.0    4.0          2    Rural  2004
7  $1,500     105068.0    5.0          1     City  2002
8  $2,291      90015.0    4.0          0    Rural  2007
9  1500.0     125976.0    2.0          0     City  2002


In [283]:
df.tail(5)

,Price,Odometer_km,Doors,Accidents,Location,Year
140,1500.0,223193.0,3.0,0,City,2003
141,1500.0,124567.0,NaN,2,Suburb,2002
142,1500.0,203153.0,4.0,0,Suburb,2004
143,1500.0,180030.0,4.0,1,City,2009
144,"$1,500",211171.0,4.0,0,??,2003


In [284]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Price        145 non-null    object 
 1   Odometer_km  138 non-null    float64
 2   Doors        138 non-null    float64
 3   Accidents    145 non-null    int64  
 4   Location     140 non-null    object 
 5   Year         145 non-null    int64  
dtypes: float64(2), int64(2), object(2)
memory usage: 6.9+ KB
None


##mission value

In [285]:
print(df.isnull().sum())

Price          0
Odometer_km    7
Doors          7
Accidents      0
Location       5
Year           0
dtype: int64


In [286]:
print(df['Location'].value_counts())

Location
City      59
Suburb    45
Rural     21
Subrb      8
??         7
Name: count, dtype: int64


## 2. Clean Target Formatting (Price)

In [287]:
df['Price'] = df['Price'].astype(str).str.replace(r'[$,]', '', regex=True).astype(float)
df

,Price,Odometer_km,Doors,Accidents,Location,Year
0,1500.0,137830.0,4.0,1,City,1998
1,4171.0,NaN,4.0,0,Rural,2016
2,5331.0,107302.0,4.0,0,Suburb,2014
3,1500.0,141838.0,4.0,1,Suburb,1999
4,1500.0,NaN,3.0,0,City,2022
...,...,...,...,...,...,...
140,1500.0,223193.0,3.0,0,City,2003
141,1500.0,124567.0,NaN,2,Suburb,2002
142,1500.0,203153.0,4.0,0,Suburb,2004
143,1500.0,180030.0,4.0,1,City,2009


### 3. Fix Category Errors before Imputation

In [288]:

df['Location'] = df['Location'].replace({'Subrb': 'Suburb', '??': np.nan})
df

,Price,Odometer_km,Doors,Accidents,Location,Year
0,1500.0,137830.0,4.0,1,City,1998
1,4171.0,NaN,4.0,0,Rural,2016
2,5331.0,107302.0,4.0,0,Suburb,2014
3,1500.0,141838.0,4.0,1,Suburb,1999
4,1500.0,NaN,3.0,0,City,2022
...,...,...,...,...,...,...
140,1500.0,223193.0,3.0,0,City,2003
141,1500.0,124567.0,NaN,2,Suburb,2002
142,1500.0,203153.0,4.0,0,Suburb,2004
143,1500.0,180030.0,4.0,1,City,2009


## 4.Handding  Missing Values

In [289]:

df['Odometer_km'] = df['Odometer_km'].fillna(df['Odometer_km'].median())
df['Doors'] = df['Doors'].fillna(df['Doors'].mode()[0])
df['Accidents'] = df['Accidents'].fillna(df['Accidents'].mode()[0])
df['Location'] = df['Location'].fillna(df['Location'].mode()[0])

##check duplicate

In [290]:
df.duplicated().sum()

np.int64(5)

##Romove Duplicate

In [291]:
df = df.drop_duplicates()
print("AFter remove duplicate:", df.shape)

AFter remove duplicate: (140, 6)


how to check outliers

In [292]:
numerical_cols = ['Price', 'Odometer_km']
for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]

    print(f"{col}: {len(outliers)} outliers")

Price: 13 outliers
Odometer_km: 2 outliers


##6. Outliers (IQR capping)

In [293]:
# 6. Outliers (IQR capping)
for col in ['Price', 'Odometer_km']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df.loc[:, col] = df[col].clip(lower=lower_bound, upper=upper_bound)

## 7. One-Hot Encode Categorical(s)

In [294]:
# One-hot encode the 'Location' column
df = pd.get_dummies(df, columns=['Location'], drop_first=True, dtype=int)

display(df.head())

,Price,Odometer_km,Doors,Accidents,Year,Location_Rural,Location_Suburb
0,1500.0,137830.0,4.0,1,1998,0,0
1,4171.0,128548.0,4.0,0,2016,1,0
2,5331.0,107302.0,4.0,0,2014,0,1
3,1500.0,141838.0,4.0,1,1999,0,1
4,1500.0,128548.0,3.0,0,2022,0,0


##8. Feature Engineering

In [295]:
df['CarAge'] = 2026 - df['Year']
df['Km_per_year'] = df.apply(lambda row: row['Odometer_km'] / row['CarAge'] if row['CarAge'] > 0 else row['Odometer_km'], axis=1)
df['Is_Urban'] = 1 - df['Location_Rural'] # If not rural, it's either City or Suburb (considered urban)
df['LogPrice'] = np.log1p(df['Price'])

display(df.head())

,Price,Odometer_km,Doors,Accidents,Year,Location_Rural,Location_Suburb,CarAge,Km_per_year,Is_Urban,LogPrice
0,1500.0,137830.0,4.0,1,1998,0,0,28,4922.500000,1,7.313887
1,4171.0,128548.0,4.0,0,2016,1,0,10,12854.800000,0,8.336151
2,5331.0,107302.0,4.0,0,2014,0,1,12,8941.833333,1,8.581482
3,1500.0,141838.0,4.0,1,1999,0,1,27,5253.259259,1,7.313887
4,1500.0,128548.0,3.0,0,2022,0,0,4,32137.000000,1,7.313887


## 9. Feature Scaling

In [296]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
cols_to_scale = ['Odometer_km', 'Year', 'CarAge', 'Km_per_year']
df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])

display(df.head())

,Price,Odometer_km,Doors,Accidents,Year,Location_Rural,Location_Suburb,CarAge,Km_per_year,Is_Urban,LogPrice
0,1500.0,0.128390,4.0,1,-1.686714,0,0,1.686714,-0.615631,1,7.313887
1,4171.0,-0.044709,4.0,0,0.794617,1,0,-0.794617,0.070446,0,8.336151
2,5331.0,-0.440923,4.0,0,0.518913,0,1,-0.518913,-0.267993,1,8.581482
3,1500.0,0.203135,4.0,1,-1.548862,0,1,1.548862,-0.587024,1,7.313887
4,1500.0,-0.044709,3.0,0,1.621727,0,0,-1.621727,1.738196,1,7.313887


## 10. Final Checks & Save

In [297]:
# 10. Final Checks & Save
print(df.isnull().sum())
df.to_csv('car_l3_clean_ready.csv', index=False)

Price              0
Odometer_km        0
Doors              0
Accidents          0
Year               0
Location_Rural     0
Location_Suburb    0
CarAge             0
Km_per_year        0
Is_Urban           0
LogPrice           0
dtype: int64
